In [27]:
def verilog_hex_to_float(hex_value, width=64, frac_bits=61):

    # Remove Verilog parts: 64'h, underscores, spaces
    hex_value = hex_value.strip().replace("_", "")

    if "'h" in hex_value.lower():
        hex_value = hex_value.lower().split("'h")[1]

    if hex_value.startswith("0x"):
        hex_value = hex_value[2:]
    # Convert hex to integer
    unsigned_value = int(hex_value, 16)
    # Convert unsigned integer to signed integer
    sign_bit = 1 << (width - 1)
    if unsigned_value & sign_bit:
        signed_value = unsigned_value - (1 << width)
    else:
        signed_value = unsigned_value
    # Convert fixed-point integer to floating-point decimal
    float_value = signed_value / (1 << frac_bits)

    return float_value




def verilog_hex_to_signed_int(hex_value, width=64):
    hex_value = hex_value.strip().replace("_", "")

    if "'h" in hex_value.lower():
        hex_value = hex_value.lower().split("'h")[1]

    if hex_value.startswith("0x"):
        hex_value = hex_value[2:]

    unsigned_value = int(hex_value, 16)

    sign_bit = 1 << (width - 1)

    if unsigned_value & sign_bit:
        return unsigned_value - (1 << width)
    else:
        return unsigned_value


def signed_int_to_verilog_hex(value, width=64):
    mask = (1 << width) - 1
    unsigned_value = value & mask
    hex_digits = width // 4

    return f"{width}'h{unsigned_value:0{hex_digits}x}"


def fixed_to_float(value, frac_bits):
    return value / (1 << frac_bits)


def multiply_q_fixed(hex_a, hex_b, width=64, frac_bits=60):
    """
    Multiply two signed fixed-point hex numbers.

    Q4.60: frac_bits=60
    Q3.61: frac_bits=61
    """

    int_a = verilog_hex_to_signed_int(hex_a, width)
    int_b = verilog_hex_to_signed_int(hex_b, width)

    # Fixed-point multiply:
    # Qm.n * Qm.n gives scale 2^(2n), so shift right by n
    product_128 = int_a * int_b
    result = product_128 >> frac_bits

    # Keep only lower 64 bits, interpreted as signed two's complement
    result_64_unsigned = result & ((1 << width) - 1)

    if result_64_unsigned & (1 << (width - 1)):
        result_64_signed = result_64_unsigned - (1 << width)
    else:
        result_64_signed = result_64_unsigned

    return {
        "input_a_signed_int": int_a,
        "input_b_signed_int": int_b,
        "input_a_float": fixed_to_float(int_a, frac_bits),
        "input_b_float": fixed_to_float(int_b, frac_bits),

        "decimal_signed_int": result_64_signed,
        "float": fixed_to_float(result_64_signed, frac_bits),

        "hex": signed_int_to_verilog_hex(result_64_signed, width),
        "binary_2s_complement": format(result_64_unsigned, f"0{width}b"),

        "product_128_signed_int": product_128,
        "result_before_64bit_wrap": result,
        "hex_128b_product": f"128'h{product_128 & ((1 << 128) - 1):032x}",
    }




In [44]:
hex_a = "64'hed83105a46afb400"
hex_b = "64'hfcf0c4e6380d7400"

print(f""" the number in float decimal with format Q3.61 is  {verilog_hex_to_float(hex_a)}""")

frac_bits = 60
result = multiply_q_fixed(hex_a, hex_b, frac_bits=frac_bits)

print(f"input in Q.{frac_bits} format")
print("Input A float:", result["input_a_float"])
print("Input B float:", result["input_b_float"])
print("Result float: ", result["float"])
print("Result hex:   ", result["hex"])

 the number in float decimal with format Q3.61 is  -0.5777509915155945
input in Q.60 format
Input A float: -1.155501983031189
Input B float: -0.19121847220154375
Result float:  0.2209533238210781
Result hex:    64'h0389065a3c107fdb


In [ ]:
import math
# comput the sin and cos and represent result in hex format into Q3.61 and Q4.60 
FRAC_BITS = 61  # Q3.61

def parse_verilog_hex(hex_str):
    s = hex_str.lower().replace("0x", "").replace("64'h", "").replace("_", "")
    val = int(s, 16)
    if val >= 2**63:
        val -= 2**64
    return val

def q_fixed_to_float(val, frac_bits=FRAC_BITS):
    return val / float(1 << frac_bits)

def float_to_q_fixed(x, frac_bits=FRAC_BITS):
    scaled = int(round(x * (1 << frac_bits)))
    if scaled >= 2**63:
        scaled -= 2**64
    elif scaled < -(2**63):
        scaled += 2**64
    return scaled

def signed64_to_hex(val):
    if val < 0:
        val = (1 << 64) + val
    return f"64'h{val:016x}"

hex_inputs = [
     "64'hed83105a46afb400",
     "64'hfcf0c4e6380d7400",
 ]

for h in hex_inputs:
    raw = parse_verilog_hex(h)
    sign_bit = 1 if raw < 0 else 0
    x = q_fixed_to_float(raw)
    s = math.sin(x)
    c = math.cos(x)
    s_q = float_to_q_fixed(s)
    c_q = float_to_q_fixed(c)
    print(f"\nInput: {h}")
    print(f"  Sign bit: {sign_bit}")
    print(f"  Sin (float): {s}")
    print(f"  Sin (hex):   {signed64_to_hex(s_q)}")

    print(f"  Cos (float): {c}")
    print(f"  Cos (hex):   {signed64_to_hex(c_q)}")



Input:  64'hed83105a46afb400
  Sign bit: 1
  Sin (float): -0.5461413408185162
  Sin (hex):   64'hee860298461bb700
  Cos (float): 0.8376930439301459
  Cos (hex):   64'h1ace61a478888b00

Input: 64'hfcf0c4e6380d7400
  Sign bit: 1
  Sin (float): -0.09546363998289319
  Sin (hex):   64'hfcf1f63c8a1f9620
  Cos (float): 0.9954329175997831
  Cos (hex):   64'h1fda96224e7e6a00
